In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
isocaproicacid,116.16,4.21197,3.507615095,259.4923136,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
isocaproicacid,H,isocaproicacid,e,2000.0,0.03
"""

model = PCSAFT(["isocaproicacid"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_isocaproicacid.csv")
fix_line_endings("rhol_isocaproicacid.csv")

Fixed: satp_isocaproicacid.csv
Fixed: rhol_isocaproicacid.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

my_liquid_density (generic function with 1 method)

In [5]:
toestimate = [
    Dict(
        :param => :segment,
        :lower => 1.0,
        :upper => 10.0,
        :guess => 3.
    ),
    Dict(
        :param => :epsilon,
        :lower => 100.,
        :upper => 500.,
        :guess => 250.
    ),
    Dict(
        :param => :sigma,
        :factor => 1e-10,
        :lower => 2.0,
        :upper => 7.0,
        :guess => 5.
    ),
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

5-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 10.0, :param => :segment, :guess => 3.0, :lower => 1.0)
 Dict(:upper => 500.0, :param => :epsilon, :guess => 250.0, :lower => 100.0)
 Dict(:factor => 1.0e-10, :param => :sigma, :upper => 7.0, :guess => 5.0, :lower => 2.0)
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.0)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_isocaproicacid.csv",
        "satp_isocaproicacid.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.7617734210263805


In [7]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([4.380644323230921, 241.13218957041283, 3.428934455125714, 2478.3054364291, 0.05550591499811987], PCSAFT{BasicIdeal, Float64}("isocaproicacid"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_isocaproicacid.csv",   my_saturation_p)


=== AAD: satp_isocaproicacid.csv ===

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



Clapeyron Estimator  exp           calc          ARD%    
382.7500    3119.740000   3156.249576   1.1703  
388.8500    4226.320000   4263.324699   0.8756  
390.7500    4679.620000   4670.455590   0.1958  
393.8500    5412.890000   5406.787573   0.1127  
398.5500    6719.450000   6713.089468   0.0947  
402.6500    8039.340000   8065.126639   0.3208  
405.7500    9452.560000   9236.115849   2.2898  
407.5500    10079.170000  9980.379491   0.9801  
408.9500    10705.790000  10594.022284  1.0440  
412.0500    12119.000000  12067.921380  0.4215  
412.1500    12225.660000  12118.225728  0.8788  
414.3500    13238.910000  13270.827604  0.2411  
414.4500    13385.570000  13325.352491  0.4499  
420.3500    16878.610000  16896.621201  0.1067  
424.9500    20051.680000  20212.861686  0.8038  
426.8500    21704.880000  21733.942361  0.1339  
432.5500    26664.470000  26885.230663  0.8279  
439.0500    33157.270000  33966.690025  2.4412  
444.0500    40330.020000  40413.806716  0.2078  
447.7500  

0.595063424747902

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_isocaproicacid.csv",   my_liquid_density)


=== AAD: rhol_isocaproicacid.csv ===
Clapeyron Estimator  exp           calc          ARD%    
273.1500    937.100000    934.083723    0.3219  
293.1400    920.900000    917.882670    0.3277  
313.1300    904.100000    902.068934    0.2247  
333.1200    887.600000    886.417298    0.1332  
353.1200    870.400000    870.720933    0.0369  
373.1200    853.700000    854.806669    0.1296  
393.1300    836.700000    838.496328    0.2147  
413.1300    819.100000    821.640697    0.3102  
433.1400    801.100000    804.054969    0.3689  
453.1400    782.400000    785.579009    0.4063  
473.1500    763.400000    765.997440    0.3402  
493.1600    743.800000    745.091006    0.1736  
513.1700    723.000000    722.582806    0.0577  
533.1700    701.100000    698.133607    0.4231  
553.1800    674.400000    671.248889    0.4672  
AARD = 0.2624%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.26238989826533854